# Lesson 02 — Otsu's Method: Automatic Threshold Finding

## Why This Lesson
Choosing the threshold value manually is guesswork. Otsu's algorithm finds the optimal threshold
automatically by maximizing the variance between the two classes (foreground and background).

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img  = cv2.imread('sample.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Manual threshold
_, manual = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

# Otsu — pass 0 as threshold, it finds it automatically
otsu_thresh, otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(f"Otsu found optimal threshold: {otsu_thresh:.0f}")

# Plot histogram with Otsu threshold marked
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(gray, cmap='gray'); axes[0].set_title('Grayscale'); axes[0].axis('off')

hist = cv2.calcHist([gray], [0], None, [256], [0,256])
axes[1].plot(hist, color='gray')
axes[1].axvline(x=127,         color='blue', linestyle='--', label='Manual = 127')
axes[1].axvline(x=otsu_thresh, color='red',  linestyle='-',  label=f'Otsu = {otsu_thresh:.0f}')
axes[1].legend(); axes[1].set_title('Histogram — where should the threshold be?')

axes[2].imshow(otsu, cmap='gray'); axes[2].set_title(f'Otsu result (threshold={otsu_thresh:.0f})'); axes[2].axis('off')
plt.tight_layout(); plt.show()

# When Otsu fails: low-contrast image
low_contrast = np.clip(gray.astype(int) // 3 + 100, 0, 255).astype(np.uint8)
_, otsu_low  = cv2.threshold(low_contrast, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Fix: CLAHE first, then Otsu
clahe      = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
enhanced   = clahe.apply(low_contrast)
_, otsu_fixed = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, im, t in zip(axes, [low_contrast, otsu_low, otsu_fixed],
    ['Low contrast input', 'Otsu fails here', 'CLAHE → Otsu (fixed)']):
    ax.imshow(im, cmap='gray'); ax.set_title(t); ax.axis('off')
plt.suptitle('Always apply CLAHE before Otsu on low-contrast images', fontsize=12)
plt.show()

## Key Takeaway
Otsu works beautifully on bimodal histograms (clear foreground + background peaks).
It fails on flat/low-contrast histograms. Fix: CLAHE first, then Otsu.